In [1]:
import sys
sys.path.append("..")

import pandas as pd

from utils import normalize_items, outlet_features, build_panel, calendar_features, prepare_forecast_data

### 1. Load & normalize items
Termasuk exclude/rename item ber-prefix `xxx.` (lihat `normalize_items.EXCLUDED_ITEMS`, `EXPLICIT_ITEM_RENAMES`) — status: penanganan kode sudah ada, tapi belum dikonfirmasi ke data owner (todo list blocker #2).

In [2]:
normalized = normalize_items.load_and_normalize()

print(normalized.shape)
normalized.head()

(693141, 7)


,Kode Barang,Tanggal,Nama Cabang,Kuantitas,Kategori Barang,Nama Barang,Satuan
0,FGS-00001,2024-01-01,KY001 - Kebuli Yaman Kutabumi (Pusat),235.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),Potong
1,FGS-00001,2024-01-01,KY002 - Kebuli Yaman Cilegon,168.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),Potong
2,FGS-00001,2024-01-01,KY003 - Kebuli Yaman Serang,84.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),Potong
3,FGS-00001,2024-01-01,KY004 - Kebuli Yaman Depok Sawangan,60.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),Potong
4,FGS-00001,2024-01-01,KY005 - Kebuli Yaman Pandeglang,121.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),Potong


**Contoh data yang butuh keputusan — item ber-prefix `xxx.`**
Sel di bawah menampilkan baris mentah (sebelum exclude/rename) untuk kode yang sudah ditangani `normalize_items.EXCLUDED_ITEMS`/`EXPLICIT_ITEM_RENAMES`, plus satu pembanding (`xxx.FGS.00068`) yang auto-merge tanpa aturan eksplisit karena namanya sudah identik dengan versi non-prefix. Tujuannya menunjukkan kenapa blocker #2 di todo list masih terbuka: keputusan exclude/rename yang ada sekarang bersifat asumsi penulis kode, belum dikonfirmasi data owner.

In [3]:
raw = pd.read_csv(normalize_items.RAW_DATA_FILE, sep=";", encoding="utf-8-sig")
raw["Tanggal"] = pd.to_datetime(raw["Tanggal"], format=normalize_items.DATE_FORMAT)

print("--- Saat ini DI-DROP oleh EXCLUDED_ITEMS ---")
excluded_rows = raw[raw["Kode Barang"].isin(normalize_items.EXCLUDED_ITEMS)]
display(excluded_rows.groupby(["Kode Barang", "Nama Barang"]).agg(
    n_baris=("Kuantitas", "size"), min_qty=("Kuantitas", "min"),
    max_qty=("Kuantitas", "max"), total_qty=("Kuantitas", "sum"),
))
display(excluded_rows.sort_values("Kuantitas", ascending=False).head(3))

print("\n--- Saat ini DI-RENAME oleh EXPLICIT_ITEM_RENAMES (xxx.FGS.00067 -> FGS-00068) ---")
renamed_rows = raw[raw["Kode Barang"] == "xxx.FGS.00067"]
display(renamed_rows.sort_values("Kuantitas", ascending=False).head(3))

print("--- Pembanding: xxx.FGS.00068 (auto-merge ke FGS-00068 tanpa aturan eksplisit, nama sudah identik) ---")
print("Catatan: xxx.FGS.00067 bernama 'Ayam Crispy Original - FG' tapi di-rename manual ke SKU")
print("'FGS-00068 / Ayam Crispy Spicy - FG' -- dua rasa berbeda digabung jadi satu SKU.")
print("Ini keputusan manual yang belum eksplisit dikonfirmasi ke data owner.")
comparison_rows = raw[raw["Kode Barang"] == "xxx.FGS.00068"]
display(comparison_rows.sort_values("Kuantitas", ascending=False).head(3))

--- Saat ini DI-DROP oleh EXCLUDED_ITEMS ---


,,n_baris,min_qty,max_qty,total_qty
Kode Barang,Nama Barang,,,,
xxx.FGS.00066,xxx.Nasi Putih,21,200.0,1600.0,9200.0
xxx.FGS.00069,xxx.Cendol Pandan - FG,74,125.0,5250.0,55500.0


,Tanggal,Kategori Barang,Kode Barang,Nama Barang,Nama Cabang,Satuan,Kuantitas
320563,2025-01-25,Barang Jadi (FG),xxx.FGS.00069,xxx.Cendol Pandan - FG,KY006 - Kebuli Yaman Depok Sentosa,Gr,5250.0
322331,2025-01-27,Barang Jadi (FG),xxx.FGS.00069,xxx.Cendol Pandan - FG,KY001 - Kebuli Yaman Kutabumi (Pusat),Gr,2625.0
320024,2025-01-25,Barang Jadi (FG),xxx.FGS.00069,xxx.Cendol Pandan - FG,KY001 - Kebuli Yaman Kutabumi (Pusat),Gr,2500.0



--- Saat ini DI-RENAME oleh EXPLICIT_ITEM_RENAMES (xxx.FGS.00067 -> FGS-00068) ---


,Tanggal,Kategori Barang,Kode Barang,Nama Barang,Nama Cabang,Satuan,Kuantitas
523644,2025-07-31,Barang Jadi (FG),xxx.FGS.00067,xxx.Ayam Crispy Original - FG,KY001 - Kebuli Yaman Kutabumi (Pusat),Potong,11.0
328409,2025-02-02,Barang Jadi (FG),xxx.FGS.00067,xxx.Ayam Crispy Original - FG,KY041 - Kebuli Yaman Sepatan,Potong,10.0
348872,2025-02-23,Barang Jadi (FG),xxx.FGS.00067,xxx.Ayam Crispy Original - FG,KY041 - Kebuli Yaman Sepatan,Potong,10.0


--- Pembanding: xxx.FGS.00068 (auto-merge ke FGS-00068 tanpa aturan eksplisit, nama sudah identik) ---
Catatan: xxx.FGS.00067 bernama 'Ayam Crispy Original - FG' tapi di-rename manual ke SKU
'FGS-00068 / Ayam Crispy Spicy - FG' -- dua rasa berbeda digabung jadi satu SKU.
Ini keputusan manual yang belum eksplisit dikonfirmasi ke data owner.


,Tanggal,Kategori Barang,Kode Barang,Nama Barang,Nama Cabang,Satuan,Kuantitas
311333,2025-01-15,Barang Jadi (FG),xxx.FGS.00068,xxx.Ayam Crispy Spicy - FG,KY001 - Kebuli Yaman Kutabumi (Pusat),Potong,12.0
363789,2025-03-09,Barang Jadi (FG),xxx.FGS.00068,xxx.Ayam Crispy Spicy - FG,KY042 - Kebuli Yaman Batavia,Potong,11.0
524579,2025-07-31,Barang Jadi (FG),xxx.FGS.00068,xxx.Ayam Crispy Spicy - FG,KY001 - Kebuli Yaman Kutabumi (Pusat),Potong,9.0


**Outlier Kuantitas ekstrem — full row (hari, outlet, barang)**
Top-20 baris Kuantitas tertinggi di seluruh `dataset.csv` mentah (lihat todo list bagian sanity-check). Kolom `is_xxx_prefix` menandai baris yang beririsan dengan keputusan `xxx.` di atas — banyak outlier ekstrem justru berasal dari item yang statusnya belum jelas, dan beberapa dalam satuan `Gr` (gram) bukan `Potong`/`Porsi`, jadi angka besar tidak otomatis berarti anomali kuantitas fisik.

In [4]:
raw["is_xxx_prefix"] = raw["Kode Barang"].str.startswith("xxx.", na=False)

cols = ["Tanggal", "Kode Barang", "Nama Barang", "Nama Cabang", "Kategori Barang", "Satuan", "Kuantitas", "is_xxx_prefix"]
top20_outliers = raw.sort_values("Kuantitas", ascending=False).head(20)[cols]

print(f"{top20_outliers['is_xxx_prefix'].sum()} dari 20 outlier teratas adalah item ber-prefix xxx.")
top20_outliers

13 dari 20 outlier teratas adalah item ber-prefix xxx.


,Tanggal,Kode Barang,Nama Barang,Nama Cabang,Kategori Barang,Satuan,Kuantitas,is_xxx_prefix
320563,2025-01-25,xxx.FGS.00069,xxx.Cendol Pandan - FG,KY006 - Kebuli Yaman Depok Sentosa,Barang Jadi (FG),Gr,5250.0,True
322331,2025-01-27,xxx.FGS.00069,xxx.Cendol Pandan - FG,KY001 - Kebuli Yaman Kutabumi (Pusat),Barang Jadi (FG),Gr,2625.0,True
320024,2025-01-25,xxx.FGS.00069,xxx.Cendol Pandan - FG,KY001 - Kebuli Yaman Kutabumi (Pusat),Barang Jadi (FG),Gr,2500.0,True
321740,2025-01-26,xxx.FGS.00069,xxx.Cendol Pandan - FG,KY042 - Kebuli Yaman Batavia,Barang Jadi (FG),Gr,2250.0,True
325166,2025-01-30,xxx.FGS.00069,xxx.Cendol Pandan - FG,KY006 - Kebuli Yaman Depok Sentosa,Barang Jadi (FG),Gr,2000.0,True
325209,2025-01-30,xxx.FGS.00069,xxx.Cendol Pandan - FG,KY001 - Kebuli Yaman Kutabumi (Pusat),Barang Jadi (FG),Gr,2000.0,True
321466,2025-01-26,xxx.FGS.00069,xxx.Cendol Pandan - FG,KY001 - Kebuli Yaman Kutabumi (Pusat),Barang Jadi (FG),Gr,2000.0,True
320323,2025-01-25,xxx.FGS.00070,xxx.Santan Cendol - FG,KY006 - Kebuli Yaman Depok Sentosa,Barang Jadi (FG),Gr,1680.0,True
328643,2025-02-02,xxx.FGS.00066,xxx.Nasi Putih,KY041 - Kebuli Yaman Sepatan,Barang Jadi (FG),Gr,1600.0,True
331949,2025-02-06,xxx.FGS.00069,xxx.Cendol Pandan - FG,KY006 - Kebuli Yaman Depok Sentosa,Barang Jadi (FG),Gr,1500.0,True


**Cek terbuka — blocker #5 (belum ada keputusan):** `reaggregate_daily` di bawah memakai `Kategori Barang: "first"`, yang diam-diam memilih kategori pertama kalau satu SKU pernah tercatat di lebih dari satu kategori (temuan `eda.ipynb` §2: 27/109 SKU). Sel berikut hanya menampilkan daftarnya sebagai pengingat, tidak mengubah perilaku pipeline.

In [5]:
categories_per_sku = normalized.groupby("Kode Barang")["Kategori Barang"].nunique()
multi_category_skus = categories_per_sku[categories_per_sku > 1]

print(f"{len(multi_category_skus)} dari {categories_per_sku.shape[0]} SKU tercatat di lebih dari satu Kategori Barang")
normalized[normalized["Kode Barang"].isin(multi_category_skus.index)].groupby("Kode Barang")["Kategori Barang"].unique()

29 dari 90 SKU tercatat di lebih dari satu Kategori Barang


Kode Barang
FGS-00001    [Barang Semi FG (WIP-2), Barang Jadi (FG)]
FGS-00002    [Barang Semi FG (WIP-2), Barang Jadi (FG)]
FGS-00003    [Barang Semi FG (WIP-2), Barang Jadi (FG)]
FGS-00004    [Barang Semi FG (WIP-2), Barang Jadi (FG)]
FGS-00005    [Barang Semi FG (WIP-2), Barang Jadi (FG)]
FGS-00006                       [Minuman, Minuman - FG]
FGS-00007                       [Minuman, Minuman - FG]
FGS-00008                       [Minuman, Minuman - FG]
FGS-00009                       [Minuman, Minuman - FG]
FGS-00012    [Barang Semi FG (WIP-2), Barang Jadi (FG)]
FGS-00013    [Barang Semi FG (WIP-2), Barang Jadi (FG)]
FGS-00014        [Barang Semi FG (WIP-2), Minuman - FG]
FGS-00015                       [Minuman, Minuman - FG]
FGS-00017                       [Minuman, Minuman - FG]
FGS-00018    [Barang Semi FG (WIP-2), Barang Jadi (FG)]
FGS-00032                       [Minuman, Minuman - FG]
FGS-00035                       [Minuman, Minuman - FG]
FGS-00037                       [Min

### 2. Outlet matching, canonicalization & region features
Filter cabang yang tidak ada di `outlets.csv` (dianggap sudah tidak beroperasi), kanonikalisasi nama cabang, lalu join `kawasan`/`hari_pengiriman` dari `dataset/outlet_mapping.csv` (blocker #1 — sekarang terbuka). `lead_time_days` untuk sementara **rata 4 hari** untuk semua kawasan (belum final per kawasan 1 vs 2, sesuai todo list item "Lead-time window 3 vs 4 hari").

In [6]:
outlets_df = outlet_features.load_outlets()
overrides_df = outlet_features.load_overrides()
region_df = outlet_features.load_region_mapping()

dropped_branches = set(normalized["Nama Cabang"].unique()) - {
    b for b in normalized["Nama Cabang"].unique()
    if outlet_features.match_branch_to_outlet(b, outlets_df, overrides_df)[0] is not None
}
print(f"{len(dropped_branches)} cabang di-drop (tidak ada di outlets.csv): {sorted(dropped_branches)}")

matched = outlet_features.filter_matched_branches(normalized, outlets_df, overrides_df)
canonical = outlet_features.canonicalize_branch_names(matched, outlets_df, overrides_df)

# Catatan: kawasan/hari_pengiriman/lead_time_days baru di-join lewat apply_region_features
# nanti setelah panel dibangun (§7) — reaggregate_daily & build_dense_panel di bawah hanya
# mempertahankan kolom di AGG_SPEC/CARRY_COLS, jadi kolom region akan hilang kalau di-join di sini.
print(canonical.shape)
canonical.head()

10 cabang di-drop (tidak ada di outlets.csv): ['KY020 - Kebuli Yaman Tambun', 'KY028 - Kebuli Yaman Condet', 'KY035 - Kebuli Yaman Antapani', 'KY046 - Kebuli Yaman Aryana Karawaci', 'KY047 - Kebuli Yaman Ciomas', 'KY052 - Kebuli Yaman Bantarjati Bogor', 'KY055 - Kebuli Yaman Ciputat Timur', 'KY059 - Kebuli Yaman Dukuh Zamrud', 'KY071 - Kebuli Yaman Citayam', 'KY072 - Kebuli Yaman Bintara']


(624290, 7)


,Kode Barang,Tanggal,Nama Cabang,Kuantitas,Kategori Barang,Nama Barang,Satuan
0,FGS-00001,2024-01-01,KY001 - Kebuli Yaman Kutabumi (Pusat),235.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),Potong
1,FGS-00001,2024-01-01,KY002 - Kebuli Yaman Cilegon,168.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),Potong
2,FGS-00001,2024-01-01,KY003 - Kebuli Yaman Serang,84.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),Potong
3,FGS-00001,2024-01-01,KY004 - Kebuli Yaman Depok Sawangan,60.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),Potong
4,FGS-00001,2024-01-01,KY005 - Kebuli Yaman Pandeglang,121.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),Potong


**Cek terbuka — blocker #3 (belum ada keputusan):** completeness tanggal per cabang terhadap rentang penuh 2024-01-01 s/d 2025-12-31. Todo list menandai KY056 (92.3%) di bawah threshold 95% — sel ini menampilkan ulang semua cabang di bawah threshold agar mudah dibawa ke data owner, tidak men-drop apa pun.

In [7]:
full_range = pd.date_range("2024-01-01", "2025-12-31", freq="D")
days_present = canonical.groupby("Nama Cabang")["Tanggal"].nunique()
completeness = (days_present / len(full_range)).sort_values()

below_threshold = completeness[completeness < 0.95]
print(f"{len(below_threshold)} cabang di bawah threshold completeness 95%:")
below_threshold

18 cabang di bawah threshold completeness 95%:


Nama Cabang
Kebuli Yaman Cilebut                       0.017784
Kebuli Yaman Cadas                         0.123119
KY067 - Kebuli Yaman Metland               0.266758
KY066 - Kebuli Yaman Parung                0.266758
KY065 - Kebuli Yaman Sangiang              0.276334
KY011 - Kebuli Yaman Bekasi Galaxy         0.310534
KY064 - Kebuli Yaman Mutiara Garuda        0.333789
KY063 - Kebuli Yaman Kedaung               0.343365
KY062 - Kebuli Yaman Kampung Baru          0.419973
KY061 - Kebuli Yaman Taman Kirana          0.419973
KY060 - Kebuli Yaman Pakuhaji              0.425445
KY058 - Kebuli Yaman Cikande               0.496580
KY057 - Kebuli Yaman Rawalumbu (Bekasi)    0.544460
KY054 - Kebuli Yaman Jagakarsa             0.716826
KY053 - Kebuli Yaman Cimanggu Bogor        0.745554
KY056 - Kebuli Yaman Tigaraksa             0.846785
KY068 - Kebuli Yaman Kramatwatu            0.900137
KY050 - Kebuli Yaman TangCity Mall         0.906977
Name: Tanggal, dtype: float64

### 3. Reaggregate & build dense panel
Reaggregate ke satu baris per (Kode Barang, Tanggal, Nama Cabang), lalu bangun panel harian padat dan buang pair dengan riwayat < `MIN_HISTORY_DAYS` (60 hari).

In [8]:
reaggregated = normalize_items.reaggregate_daily(canonical)
panel = build_panel.build_dense_panel(reaggregated)

panel_before_filter_shape = panel.shape
panel = build_panel.filter_min_history(panel)

print(f"panel sebelum filter min-history: {panel_before_filter_shape}")
print(f"panel setelah filter min-history: {panel.shape}")
panel.head()

panel sebelum filter min-history: (1357631, 6)
panel setelah filter min-history: (1341346, 6)


,Kode Barang,Nama Cabang,Tanggal,Kuantitas,Kategori Barang,Nama Barang
0,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-01,235.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9)
1,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-02,147.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9)
2,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-03,85.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9)
3,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-04,106.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9)
4,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-05,155.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9)


**Cek terbuka — blocker #4 (belum ada keputusan):** breakdown heuristik kenapa pair gagal `MIN_HISTORY_DAYS` — SKU baru (kemunculan pertama SKU di seluruh dataset dekat cutoff), Outlet baru (idem untuk cabang), atau lainnya (kemungkinan data hilang di pair yang SKU & cabangnya sama-sama sudah lama ada). Ini heuristik untuk mempersempit investigasi, bukan keputusan final.

In [9]:
cutoff = build_panel.TEST_START
min_days = build_panel.MIN_HISTORY_DAYS
new_cutoff = cutoff - pd.Timedelta(days=min_days)

pre_cutoff_counts = (
    reaggregated[reaggregated["Tanggal"] < cutoff]
    .groupby(build_panel.PAIR_COLS)
    .size()
    .reset_index(name="pre_cutoff_days")
)
failing_pairs = pre_cutoff_counts[pre_cutoff_counts["pre_cutoff_days"] < min_days].copy()

sku_first_date = reaggregated.groupby("Kode Barang")["Tanggal"].min()
branch_first_date = reaggregated.groupby("Nama Cabang")["Tanggal"].min()
failing_pairs["sku_first_date"] = failing_pairs["Kode Barang"].map(sku_first_date)
failing_pairs["branch_first_date"] = failing_pairs["Nama Cabang"].map(branch_first_date)

def categorize(row):
    if row["sku_first_date"] >= new_cutoff:
        return "SKU baru"
    if row["branch_first_date"] >= new_cutoff:
        return "Outlet baru"
    return "Lainnya (kemungkinan data hilang)"

failing_pairs["kategori"] = failing_pairs.apply(categorize, axis=1)

print(f"{len(failing_pairs)} dari {len(pre_cutoff_counts)} pair gagal MIN_HISTORY_DAYS ({min_days} hari)")
failing_pairs["kategori"].value_counts()

1285 dari 3097 pair gagal MIN_HISTORY_DAYS (60 hari)


kategori
Lainnya (kemungkinan data hilang)    1102
SKU baru                              143
Outlet baru                            40
Name: count, dtype: int64

### 4. Targets, lag & rolling features

In [10]:
featured = prepare_forecast_data.add_targets(panel)
featured = prepare_forecast_data.add_lag_features(featured)
featured = prepare_forecast_data.add_rolling_features(featured)

print(featured.shape)
featured.head()

(1341346, 26)


,Kode Barang,Nama Cabang,Tanggal,Kuantitas,Kategori Barang,Nama Barang,target_h1,target_h2,target_h3,target_h4,...,lag_7,lag_14,lag_21,lag_28,roll_mean_7,roll_std_7,roll_mean_14,roll_std_14,roll_mean_28,roll_std_28
0,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-01,235.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),147.0,85.0,106.0,155.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-02,147.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),85.0,106.0,155.0,240.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-03,85.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),106.0,155.0,240.0,235.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-04,106.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),155.0,240.0,235.0,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-05,155.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),240.0,235.0,100.0,77.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 5. Calendar features

In [11]:
calendar_features.check_year_coverage(featured["Tanggal"])
featured = calendar_features.add_calendar_features(featured)

print(featured.shape)
featured.head()

(1341346, 46)


,Kode Barang,Nama Cabang,Tanggal,Kuantitas,Kategori Barang,Nama Barang,target_h1,target_h2,target_h3,target_h4,...,days_until_eid_al_fitr,is_eid_al_adha,days_since_eid_al_adha,days_until_eid_al_adha,is_independence_day,days_since_independence_day,days_until_independence_day,is_new_year,days_since_new_year,days_until_new_year
0,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-01,235.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),147.0,85.0,106.0,155.0,...,NaN,False,NaN,NaN,False,NaN,NaN,True,0.0,0.0
1,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-02,147.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),85.0,106.0,155.0,240.0,...,NaN,False,NaN,NaN,False,NaN,NaN,False,1.0,NaN
2,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-03,85.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),106.0,155.0,240.0,235.0,...,NaN,False,NaN,NaN,False,NaN,NaN,False,2.0,NaN
3,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-04,106.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),155.0,240.0,235.0,100.0,...,NaN,False,NaN,NaN,False,NaN,NaN,False,3.0,NaN
4,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-05,155.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),240.0,235.0,100.0,77.0,...,NaN,False,NaN,NaN,False,NaN,NaN,False,4.0,NaN


### 6. Branch stats, outlet features & region/lead-time features
`compute_branch_stats` pakai cutoff (train-only, no leakage). `apply_outlet_features` join `kota`/`has_shopee`/`has_gofood`/`has_grabfood`/`can_order_online` (statis per cabang). `apply_region_features` join `kawasan`/`hari_pengiriman` dari `outlet_mapping.csv` + `lead_time_days` (rata 4 hari untuk semua kawasan, sementara).

In [12]:
branch_stats = prepare_forecast_data.compute_branch_stats(featured)
featured = prepare_forecast_data.apply_branch_stats(featured, branch_stats)
featured = prepare_forecast_data.add_branch_age_days(featured)
featured = prepare_forecast_data.apply_outlet_features(featured, outlets_df, overrides_df)
featured = outlet_features.apply_region_features(featured, region_df)

print(featured.shape)
featured.head()

(1341346, 58)


,Kode Barang,Nama Cabang,Tanggal,Kuantitas,Kategori Barang,Nama Barang,target_h1,target_h2,target_h3,target_h4,...,branch_volume_tier,branch_age_days,kota,has_shopee,has_gofood,has_grabfood,can_order_online,kawasan,hari_pengiriman,lead_time_days
0,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-01,235.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),147.0,85.0,106.0,155.0,...,flagship,0,Kota Tangerang,True,True,True,True,1,Senin dan Kamis,4
1,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-02,147.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),85.0,106.0,155.0,240.0,...,flagship,1,Kota Tangerang,True,True,True,True,1,Senin dan Kamis,4
2,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-03,85.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),106.0,155.0,240.0,235.0,...,flagship,2,Kota Tangerang,True,True,True,True,1,Senin dan Kamis,4
3,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-04,106.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),155.0,240.0,235.0,100.0,...,flagship,3,Kota Tangerang,True,True,True,True,1,Senin dan Kamis,4
4,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-05,155.0,Barang Semi FG (WIP-2),Ayam Kebuli (0.9),240.0,235.0,100.0,77.0,...,flagship,4,Kota Tangerang,True,True,True,True,1,Senin dan Kamis,4


In [13]:
# QA: setiap baris di panel harus dapat kawasan (tidak ada NaN dari region join)
unmatched_region = featured[featured["kawasan"].isna()]["Nama Cabang"].unique()
assert len(unmatched_region) == 0, f"Cabang tanpa kawasan: {unmatched_region}"

print(featured.groupby("kawasan")["Nama Cabang"].nunique())
print(featured["lead_time_days"].value_counts())

kawasan
1    12
2    40
Name: Nama Cabang, dtype: int64
lead_time_days
4    1341346
Name: count, dtype: int64


### 7. QA checks

In [14]:
# Kuantitas non-negatif — lihat catatan CLAUDE.md soal anomali KY011 2024-02-29 (belum dikonfirmasi ulang ke data owner)
assert (featured["Kuantitas"] >= 0).all(), "Ditemukan Kuantitas negatif"

# Tidak ada baris duplikat per (pair, tanggal)
dup_count = featured.duplicated(subset=build_panel.PAIR_COLS + ["Tanggal"]).sum()
assert dup_count == 0, f"Ditemukan {dup_count} baris duplikat (pair, Tanggal)"

print("Kuantitas non-negatif: OK")
print("Tidak ada duplikat (pair, Tanggal): OK")

Kuantitas non-negatif: OK
Tidak ada duplikat (pair, Tanggal): OK


In [15]:
# Leakage spot-check: lag_1 baris tertentu harus sama dengan Kuantitas H-1 pada pair yang sama
sample = featured[featured["lag_1"].notna()].sample(1, random_state=0).iloc[0]
prior_day = sample["Tanggal"] - pd.Timedelta(days=1)
prior_row = featured[
    (featured["Kode Barang"] == sample["Kode Barang"])
    & (featured["Nama Cabang"] == sample["Nama Cabang"])
    & (featured["Tanggal"] == prior_day)
]
assert not prior_row.empty, "Baris H-1 untuk sample tidak ditemukan"
assert sample["lag_1"] == prior_row.iloc[0]["Kuantitas"], "lag_1 tidak cocok dengan Kuantitas H-1"

print(f"Leakage spot-check OK — pair={sample['Kode Barang']}/{sample['Nama Cabang']}, tanggal={sample['Tanggal'].date()}")

Leakage spot-check OK — pair=FGS-00014/KY005 - Kebuli Yaman Pandeglang, tanggal=2024-08-24


In [16]:
# Outlet-match QA: tidak ada cabang "Unknown" (unmatched sudah di-drop di §2)
assert (featured["kota"] != "Unknown").all(), "Ditemukan cabang dengan kota Unknown"

# Setiap Nama Cabang harus map ke tepat satu kota (tidak ada fan-out dari join)
kota_per_branch = featured.groupby("Nama Cabang")["kota"].nunique()
assert (kota_per_branch == 1).all(), "Ditemukan cabang dengan lebih dari satu kota"

print("Outlet-match QA: OK")
print(featured["can_order_online"].value_counts(dropna=False))

Outlet-match QA: OK
can_order_online
True     1325890
False      15456
Name: count, dtype: int64


In [17]:
featured.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1341346 entries, 0 to 1341345
Data columns (total 58 columns):
 #   Column                       Non-Null Count    Dtype         
---  ------                       --------------    -----         
 0   Kode Barang                  1341346 non-null  object        
 1   Nama Cabang                  1341346 non-null  object        
 2   Tanggal                      1341346 non-null  datetime64[ns]
 3   Kuantitas                    1341346 non-null  float64       
 4   Kategori Barang              1341346 non-null  object        
 5   Nama Barang                  1341346 non-null  object        
 6   target_h1                    1338710 non-null  float64       
 7   target_h2                    1336074 non-null  float64       
 8   target_h3                    1333438 non-null  float64       
 9   target_h4                    1330802 non-null  float64       
 10  target_h5                    1328166 non-null  float64       
 11  target_h6  

---
**Berhenti di sini untuk sementara** — `split_train_test`/`export_splits` belum dijalankan. Cek dulu hasil tiap sel di atas (terutama 4 cek blocker yang masih terbuka: xxx. prefix, KY056 completeness, breakdown 842 pair, kategori time-varying) sebelum lanjut ke pemisahan train/test & export ke `dataset/model_ready/`.